RESULTS - Aprendizaje Profundo con YOLO
==========================================================

Este cuaderno carga los resultados de la evaluación generados previamente (`evaluation_results.csv`) y genera visualizaciones para comparar el rendimiento de las distintas variantes de YOLOv8 (nano, small, medium, large, x-large).

Se analizarán las siguientes dimensiones:
1.  **Precisión (F1-Score):** ¿Qué tan bien detecta los objetos?
2.  **Velocidad (Tiempo de Inferencia):** ¿Qué tan rápido procesa las imágenes?
3.  **Eficiencia (Trade-off):** Relación entre calidad y coste computacional.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Crear carpeta para guardar las imágenes
output_dir = 'resultados'
os.makedirs(output_dir, exist_ok=True)
print(f"Trabajando con directorio: '{output_dir}'")

# Cargar resultados
csv_file = os.path.join(output_dir, 'evaluation_results.csv')
if not os.path.exists(csv_file):
    print(f"Error: No se encuentra {csv_file}. Ejecuta primero eval.ipynb.")
else:
    df = pd.read_csv(csv_file)
    print("Tabla de Resultados cargada correctamente:")
    display(df)

### Comparativa de Precisión (F1-Score)
Observamos cómo varía la calidad de la detección al aumentar el tamaño del modelo. Generalmente, modelos más grandes tienen mayor F1.

In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Crear gráfico de barras
bars = plt.bar(df['variant'], df['f1'], color='skyblue', edgecolor='black')

# Añadir etiquetas de valor encima de las barras
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom')

plt.xlabel('Variante del Modelo (YOLOv8)')
plt.ylabel('F1-Score')
plt.title('Comparación de Precisión (F1-Score) por Modelo')
plt.ylim(0, 1.1)

# Guardar imagen
save_path = os.path.join(output_dir, 'f1_score_comparison.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Gráfica guardada en: {save_path}")

plt.show()

### Comparativa de Velocidad (Tiempo de Inferencia)
Aquí comparamos cuánto tarda en milisegundos cada modelo en procesar una imagen. Los modelos más grandes requieren más cómputo.

In [ ]:
plt.figure(figsize=(10, 6))

# Crear gráfico de barras
bars = plt.bar(df['variant'], df['time_ms'], color='salmon', edgecolor='black')

# Añadir etiquetas
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 2,
             f'{height:.1f} ms', ha='center', va='bottom')

plt.xlabel('Variante del Modelo')
plt.ylabel('Tiempo de Inferencia (ms)')
plt.title('Tiempo Promedio de Inferencia por Imagen')

# Guardar imagen
save_path = os.path.join(output_dir, 'inference_time_comparison.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Gráfica guardada en: {save_path}")

plt.show()

### Trade-off: Velocidad vs. Precisión
Este gráfico de dispersión ("scatter plot") es crucial para elegir el modelo adecuado.
* **Eje X (Tiempo):** Cuanto más a la derecha, más lento.
* **Eje Y (F1):** Cuanto más arriba, mejor detección.
* **Tamaño del punto:** Proporcional al número de parámetros del modelo.

In [ ]:
plt.figure(figsize=(12, 8))

# Escalar el tamaño de los puntos para que se vean bien (basado en params)
# Normalizamos un poco para que el punto 'n' no sea invisible
sizes = df['params'] / df['params'].max() * 1000 + 100

plt.scatter(df['time_ms'], df['f1'], s=sizes, alpha=0.6, c='purple', edgecolors='black')

# Añadir etiquetas con el nombre de cada modelo
for i, row in df.iterrows():
    plt.text(row['time_ms'], row['f1'], f" YOLOv8{row['variant']}", 
             fontsize=12, ha='left', va='bottom', weight='bold')

plt.xlabel('Tiempo de Inferencia (ms) [Menos es más rápido]')
plt.ylabel('F1-Score [Más es mejor]')
plt.title('Trade-off: Velocidad vs Precisión (El tamaño del punto indica el peso del modelo)')
plt.grid(True, linestyle='--', alpha=0.7)

# Guardar imagen
save_path = os.path.join(output_dir, 'tradeoff_speed_precision.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Gráfica guardada en: {save_path}")

plt.show()

### Exportación a LaTeX
A continuación generamos el código LaTeX de la tabla de resultados para copiarlo y pegarlo directamente en el informe del proyecto.

In [ ]:
# Seleccionamos las columnas más relevantes para el informe
cols_to_print = ['model', 'f1', 'precision', 'recall', 'time_ms', 'params']
header = ['Modelo', 'F1-Score', 'Precisión', 'Recall', 'Tiempo (ms)', 'Parámetros']

# Generar código LaTeX
latex_code = df[cols_to_print].to_latex(
    index=False,                  # No imprimir el índice de pandas (0, 1, 2...)
    header=header,                # Nombres de columnas más bonitos
    float_format="%.3f",          # Formato para decimales (0.000)
    caption="Comparativa de rendimiento de los modelos YOLOv8 en el dataset de validación.",
    label="tab:resultados_yolo",  # Etiqueta para referenciarla en el texto con \ref{tab:resultados_yolo}
    column_format="lccccc"        # Alineación: left, center, center...
)

print("-" * 20)
print("COPIA EL CÓDIGO DE ABAJO:")
print("-" * 20)
print(latex_code)